In [1]:
import numpy as np
import pandas as pd
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, SliceEvaluator, evaluate_slice_by_slice, save_evaluation_results
from pathlib import Path


root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

methods_available = ["raycast", "local_normals"]
propagation_styles = ['default', 'full', 'prompt_based', 'central_start', 'central_partitions']
method = methods_available[1]
propagation_style = propagation_styles[3]

rootpath = Path(root)
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()])
print(subjects)

slice_evals = []


['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']


In [2]:
for subject_nr in range(len(subjects)):
    data = DataLoader(parentfolder=root,subject_nr=subject_nr,volume_of_interest="CTVT",verbose=True)
    unc_handler = UG_prompter(data=data)
    seg_handler = Segmentation(data=data)

    unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=3, method="raycast", mode="median") #unc_threshold=0.033470
    unc_handler.compute_band_thickness(method=method)

    nietjes_prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=4.0,
    interpix_dist=3,
    pixel_interval=10,
    angle_step=5,
    method=method)

    bbox_prompts = unc_handler.generate_prompts_boxes(band_threshold=0.0)

    dense_prompt = seg_handler.load_dense_prompt()
    dense_and_nietjes_prompts = combine_prompt_sets(prompt_dict_list = [dense_prompt, nietjes_prompts])

    prompt_sets = [dense_and_nietjes_prompts, bbox_prompts]
    prompt_names = ["Dense_and_nietjes", "Uncertainty_bboxes"]
    
    prompt_weights = [0.2, 0.8]

    seg_handler.compile_prompt_sets(prompt_dict_list=prompt_sets, prompt_set_names=prompt_names, prompt_set_weights=prompt_weights)

    seg_handler.run_segmentation_sets(propagation_style=propagation_style, weighting_strategy="custom", threshold=0.0)
    seg_handler.remove_distant_slices(tolerance_frames=0)

    slice_results_df = evaluate_slice_by_slice(
    pred=seg_handler.predicted_seg,
    gt=data.gt,
    uncertainty=unc_handler.unc_map_bin,
    spacing=data.img_spacing,
    subject_name=data.subject_name,
    surface_dice_tol=1.0,
    )
    
    # Store your generated segmentation results
    slice_results_df["method"] = "MedSAM_prediction"
    slice_evals.append(slice_results_df)


    # Store nnUNet slice-based results
    nnunet_slice_results_df = evaluate_slice_by_slice(
        pred=seg_handler.mask,
        gt=data.gt,
        uncertainty=None,
        spacing=data.img_spacing,
        subject_name=data.subject_name,
        surface_dice_tol=1.0,
    )

    nnunet_slice_results_df["method"] = "nnUNet"
    slice_evals.append(nnunet_slice_results_df)


    # Store observer/recontour slice-based results
    observer_names = ["B", "C", "D", "E"]
    data.load_recontours()
    for observer_name, observer_recontour in zip(observer_names, data.observer_recontours):
        observer_slice_results_df = evaluate_slice_by_slice(
            pred=observer_recontour,
            gt=data.gt,
            uncertainty=None,
            spacing=data.img_spacing,
            subject_name=data.subject_name,
            surface_dice_tol=1.0,
        )

        observer_slice_results_df["method"] = f"Observer {observer_name}"
        slice_evals.append(observer_slice_results_df)


    # Combine everything into one dataframe
    all_slice_results_df = pd.concat(slice_evals, ignore_index=True)

slice_results_df

all_slice_results_df.to_csv(
    "slice_based_results.csv",
    index=False
)

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


iter=00 | thr=0.194962 | band=0.94 mm | error=2.06
iter=01 | thr=0.097481 | band=1.41 mm | error=1.59
iter=02 | thr=0.048740 | band=1.99 mm | error=1.01
iter=03 | thr=0.024370 | band=2.46 mm | error=0.54
iter=04 | thr=0.012185 | band=2.93 mm | error=0.07
iter=05 | thr=0.006093 | band=3.63 mm | error=0.63
iter=06 | thr=0.009139 | band=3.28 mm | error=0.28
iter=07 | thr=0.010662 | band=3.16 mm | error=0.16
iter=08 | thr=0.011424 | band=3.05 mm | error=0.05
iter=09 | thr=0.011804 | band=3.05 mm | error=0.05
iter=10 | thr=0.011995 | band=3.05 mm | error=0.05
iter=11 | thr=0.012090 | band=2.93 mm | error=0.07
iter=12 | thr=0.012042 | band=2.93 mm | error=0.07
iter=13 | thr=0.012019 | band=2.99 mm | error=0.01
iter=14 | thr=0.012007 | band=2.99 mm | error=0.01
iter=15 | thr=0.012001 | band=2.99 mm | error=0.01
iter=16 | thr=0.011998 | band=3.05 mm | error=0.05
iter=17 | thr=0.011999 | band=2.99 mm | error=0.01
iter=18 | thr=0.011998 | band=2.99 mm | error=0.01
iter=19 | thr=0.011998 | band=3

c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (from middle slice 36)...


...

Running segmentation for prompt set 'Uncertainty_bboxes' with slices: [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Adding prompt(s) on slice 45
Adding prompt(s) on slice 46
Adding prompt(s) on slice 47
Adding prompt(s) on slice 48
Adding prompt(s) on slice 49
Adding prompt(s) on slice 50
Adding prompt(s) on slice 51
Forward propagation (from middle slice 42)...


propagate in video: 100%|██████████| 46/46 [00:11<00:00,  3.97it/s]


Backward propagation (from middle slice 42)...


propagate in video: 100%|██████████| 43/43 [00:10<00:00,  4.16it/s]


Kept slices 33 to 51. Removed predicted segmentation outside dense mask ±0 slices.
Evaluating slices 33 to 51...
Number of evaluated slices: 19
Evaluating slices 33 to 51...
Number of evaluated slices: 19
Evaluating slices 34 to 51...
Number of evaluated slices: 18
Evaluating slices 33 to 51...
Number of evaluated slices: 19
Evaluating slices 33 to 50...
Number of evaluated slices: 18
Evaluating slices 33 to 50...
Number of evaluated slices: 18


In [2]:
#CELL TO CREATE INTERACTIVE TABLE TO EXPLORE SLICE-BASED RESULTS

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output


def explore_slice_results(csv_path):
    df = pd.read_csv(csv_path)

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [c for c in df.columns if c not in numeric_cols]

    metric_select = widgets.SelectMultiple(
        options=numeric_cols,
        value=tuple(numeric_cols[: min(5, len(numeric_cols))]),
        description="Metrics:",
        rows=8
    )

    group_dropdown = widgets.Dropdown(
        options=["None"] + categorical_cols,
        value="method" if "method" in categorical_cols else "None",
        description="Group:"
    )

    group_values_select = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Show:",
        rows=6
    )

    stat_select = widgets.SelectMultiple(
        options=[
            "count",
            "mean",
            "std",
            "median",
            "min",
            "max",
            "missing",
            "missing_percent",
        ],
        value=("count", "mean", "std", "median", "min", "max"),
        description="Stats:",
        rows=8
    )

    out = widgets.Output()

    def update_group_values(change=None):
        group_col = group_dropdown.value

        if group_col == "None":
            group_values_select.options = []
            group_values_select.value = ()
            group_values_select.disabled = True
        else:
            values = sorted(df[group_col].dropna().astype(str).unique().tolist())
            group_values_select.options = values
            group_values_select.value = tuple(values)
            group_values_select.disabled = False

        update_table()

    def get_filtered_df():
        group_col = group_dropdown.value

        if group_col == "None":
            return df.copy()

        selected_groups = list(group_values_select.value)

        if len(selected_groups) == 0:
            return df.iloc[0:0].copy()

        return df[df[group_col].astype(str).isin(selected_groups)].copy()

    def summarize_numeric(data, selected_metrics, selected_stats):
        rows = []

        for metric in selected_metrics:
            values = data[metric]

            row = {"metric": metric}

            if "count" in selected_stats:
                row["count"] = values.count()

            if "mean" in selected_stats:
                row["mean"] = values.mean()

            if "std" in selected_stats:
                row["std"] = values.std()

            if "median" in selected_stats:
                row["median"] = values.median()

            if "min" in selected_stats:
                row["min"] = values.min()

            if "max" in selected_stats:
                row["max"] = values.max()

            if "missing" in selected_stats:
                row["missing"] = values.isna().sum()

            if "missing_percent" in selected_stats:
                row["missing_percent"] = 100 * values.isna().mean()

            rows.append(row)

        return pd.DataFrame(rows)

    def update_table(change=None):
        with out:
            clear_output(wait=True)

            plot_df = get_filtered_df()
            selected_metrics = list(metric_select.value)
            selected_stats = list(stat_select.value)
            group_col = group_dropdown.value

            if plot_df.empty:
                print("No data selected.")
                return

            if len(selected_metrics) == 0:
                print("Select at least one metric.")
                return

            if len(selected_stats) == 0:
                print("Select at least one statistic.")
                return

            if group_col == "None":
                summary_df = summarize_numeric(
                    plot_df,
                    selected_metrics,
                    selected_stats
                )
            else:
                summaries = []

                for group_name, group_df in plot_df.groupby(group_col):
                    group_summary = summarize_numeric(
                        group_df,
                        selected_metrics,
                        selected_stats
                    )
                    group_summary.insert(0, group_col, group_name)
                    summaries.append(group_summary)

                summary_df = pd.concat(summaries, ignore_index=True)

            display(summary_df)

    metric_select.observe(update_table, names="value")
    stat_select.observe(update_table, names="value")
    group_dropdown.observe(update_group_values, names="value")
    group_values_select.observe(update_table, names="value")

    display(
        widgets.VBox([
            widgets.HBox([group_dropdown]),
            group_values_select,
            widgets.HBox([metric_select, stat_select]),
            out
        ])
    )

    update_group_values()

In [3]:
explore_slice_results("Slice_based_results.csv")

In [4]:
# CELL FOR INTERACTIVE PLOTTER TO EXPLORE SLICE-BASED RESULTS

import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import gaussian_kde
import plotly.graph_objects as go


def analyze_slice_results(csv_path):
    df = pd.read_csv(csv_path)

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [c for c in df.columns if c not in numeric_cols]

    if "method" not in df.columns:
        raise ValueError("This function expects a 'method' column in the dataframe.")

    subject_col = None
    if "subject" in df.columns:
        subject_col = "subject"
    elif "subject_name" in df.columns:
        subject_col = "subject_name"
    elif "subject name" in df.columns:
        subject_col = "subject name"

    if subject_col is None:
        raise ValueError("Could not find a subject column. Expected 'subject', 'subject_name', or 'subject name'.")

    method_values = sorted(df["method"].dropna().astype(str).unique().tolist())
    subject_values = sorted(df[subject_col].dropna().astype(str).unique().tolist())

    x_dropdown = widgets.Dropdown(
        options=numeric_cols,
        value="relative slice idx" if "relative slice idx" in numeric_cols else numeric_cols[0],
        description="X:"
    )

    y_dropdown = widgets.Dropdown(
        options=numeric_cols,
        value="SurfaceDice@1.0mm" if "SurfaceDice@1.0mm" in numeric_cols else numeric_cols[1],
        description="Y:"
    )

    group_dropdown = widgets.Dropdown(
        options=["None"] + categorical_cols,
        value="method" if "method" in categorical_cols else "None",
        description="Group:"
    )

    method_values_select = widgets.SelectMultiple(
        options=method_values,
        value=tuple(method_values),
        description="Methods:",
        rows=min(8, max(3, len(method_values)))
    )

    subject_values_select = widgets.SelectMultiple(
        options=subject_values,
        value=tuple(subject_values),
        description="Subjects:",
        rows=min(8, max(3, len(subject_values)))
    )

    group_values_select = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Groups:",
        rows=6
    )

    plot_dropdown = widgets.Dropdown(
        options=["Scatter", "Boxplot", "Histogram", "Density"],
        value="Scatter",
        description="Plot:"
    )

    out = widgets.Output()

    def update_group_values(change=None):
        group_col = group_dropdown.value

        if group_col == "None" or group_col in ["method", subject_col]:
            group_values_select.options = []
            group_values_select.value = ()
            group_values_select.disabled = True
        else:
            values = sorted(df[group_col].dropna().astype(str).unique().tolist())
            group_values_select.options = values
            group_values_select.value = tuple(values)
            group_values_select.disabled = False

        update_plot()

    def get_filtered_df():
        plot_df = df.copy()

        selected_methods = list(method_values_select.value)
        selected_subjects = list(subject_values_select.value)

        if len(selected_methods) == 0 or len(selected_subjects) == 0:
            return df.iloc[0:0].copy()

        plot_df = plot_df[plot_df["method"].astype(str).isin(selected_methods)]
        plot_df = plot_df[plot_df[subject_col].astype(str).isin(selected_subjects)]

        group_col = group_dropdown.value

        if group_col != "None" and group_col not in ["method", subject_col]:
            selected_groups = list(group_values_select.value)

            if len(selected_groups) == 0:
                return df.iloc[0:0].copy()

            plot_df = plot_df[plot_df[group_col].astype(str).isin(selected_groups)]

        return plot_df.copy()

    def update_plot(change=None):
        with out:
            clear_output(wait=True)

            plot_df = get_filtered_df()

            x_col = x_dropdown.value
            y_col = y_dropdown.value
            group_col = group_dropdown.value
            plot_type = plot_dropdown.value

            color = None if group_col == "None" else group_col

            if plot_df.empty:
                print("No data selected.")
                return

            if plot_type == "Scatter":
                fig = px.scatter(
                    plot_df,
                    x=x_col,
                    y=y_col,
                    color=color,
                    hover_data=plot_df.columns,
                    title=f"{y_col} vs {x_col}"
                )

            elif plot_type == "Boxplot":
                if group_col == "None":
                    fig = px.box(
                        plot_df,
                        y=y_col,
                        points="all",
                        title=f"Boxplot of {y_col}"
                    )
                else:
                    fig = px.box(
                        plot_df,
                        x=group_col,
                        y=y_col,
                        color=group_col,
                        points="all",
                        title=f"{y_col} grouped by {group_col}"
                    )

            elif plot_type == "Histogram":
                fig = px.histogram(
                    plot_df,
                    x=x_col,
                    color=color,
                    barmode="overlay",
                    opacity=0.6,
                    marginal="box",
                    title=f"Histogram of {x_col}"
                )

            elif plot_type == "Density":
                fig = go.Figure()

                if group_col == "None":
                    values = plot_df[x_col].dropna()

                    if len(values) >= 3 and values.nunique() > 1:
                        kde = gaussian_kde(values)
                        xs = np.linspace(values.min(), values.max(), 300)

                        fig.add_trace(
                            go.Scatter(
                                x=xs,
                                y=kde(xs),
                                mode="lines",
                                name="All"
                            )
                        )

                else:
                    for group_name, group_df in plot_df.groupby(group_col):
                        values = group_df[x_col].dropna()

                        if len(values) < 3 or values.nunique() <= 1:
                            continue

                        kde = gaussian_kde(values)
                        xs = np.linspace(values.min(), values.max(), 300)

                        fig.add_trace(
                            go.Scatter(
                                x=xs,
                                y=kde(xs),
                                mode="lines",
                                name=str(group_name)
                            )
                        )

                fig.update_layout(
                    title=f"Density plot of {x_col}",
                    xaxis_title=x_col,
                    yaxis_title="Density"
                )

            fig.show()

    x_dropdown.observe(update_plot, names="value")
    y_dropdown.observe(update_plot, names="value")
    group_dropdown.observe(update_group_values, names="value")
    method_values_select.observe(update_plot, names="value")
    subject_values_select.observe(update_plot, names="value")
    group_values_select.observe(update_plot, names="value")
    plot_dropdown.observe(update_plot, names="value")

    display(
        widgets.VBox([
            widgets.HBox([plot_dropdown, group_dropdown]),
            widgets.HBox([x_dropdown, y_dropdown]),
            widgets.HBox([method_values_select, subject_values_select, group_values_select]),
            out
        ])
    )

    update_group_values()

In [ ]:
analyze_slice_results(
    "slice_based_results.csv"
)